# 🎮 Jump & Run

### Controls
- **Arrow Keys / WASD** — Move
- **Space / Up / W** — Jump
- **F** — Attack (Lightning / Sonic Wave)
- **Stomp enemies** from above!

### 🕹️ Play the Game

In [ ]:
# Play the game (V2 with player name support)
game_html = open("./builtin/jump_and_run_v2.html").read()
displayHTML(game_html)

### 💾 Save Stats
Copy the JSON from the game output, run the script below and paste it in, and click Save.

In [ ]:
import json
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display as ipy_display
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    BooleanType, FloatType, TimestampType
)

schema = StructType([
    StructField("player_name", StringType(), True),
    StructField("timestamp", TimestampType(), True),
    StructField("duration_seconds", FloatType(), True),
    StructField("score", IntegerType(), True),
    StructField("won", BooleanType(), True),
    StructField("lives_left", IntegerType(), True),
    StructField("deaths_total", IntegerType(), True),
    StructField("deaths_enemy", IntegerType(), True),
    StructField("deaths_water", IntegerType(), True),
    StructField("deaths_fall", IntegerType(), True),
    StructField("deaths_lava", IntegerType(), True),
    StructField("coins_collected", IntegerType(), True),
    StructField("enemies_stomped", IntegerType(), True),
    StructField("enemies_zapped", IntegerType(), True),
    StructField("bosses_killed", IntegerType(), True),
    StructField("attacks_used", IntegerType(), True),
    StructField("jumps", IntegerType(), True),
    StructField("forms_collected", StringType(), True),
    StructField("final_form", StringType(), True),
    StructField("max_x_reached", IntegerType(), True),
    StructField("level_reached", IntegerType(), True),
])

textarea = widgets.Textarea(
    placeholder='Paste the stats JSON here (Ctrl+V)...',
    layout=widgets.Layout(width='100%', height='80px')
)
btn = widgets.Button(description='💾 Save Stats', button_style='primary')
output = widgets.Output()

def on_save(b):
    output.clear_output()
    with output:
        txt = textarea.value.strip()
        if not txt or not txt.startswith("{"):
            print("⚠️ Paste valid JSON first.")
            return
        raw = json.loads(txt)
        raw["timestamp"] = datetime.fromisoformat(raw["timestamp"].replace("Z", "+00:00"))
        raw["duration_seconds"] = float(raw["duration_seconds"])
        for f in schema.fields:
            if f.name not in raw:
                raw[f.name] = None
        row = [tuple(raw[f.name] for f in schema.fields)]
        df = spark.createDataFrame(row, schema=schema)
        if not spark.catalog.tableExists("game_stats"):
            df.write.format("delta").saveAsTable("game_stats")
        else:
            df.write.mode("append").option("mergeSchema", "true").format("delta").saveAsTable("game_stats")
        print(f"✅ Saved! Player: {raw.get('player_name', 'Unknown')} | Score: {raw['score']} | Won: {raw['won']} | L{raw.get('level_reached', '?')} | Duration: {raw['duration_seconds']}s")
        textarea.value = ""

btn.on_click(on_save)
ipy_display(widgets.VBox([widgets.Label('📋 Paste game stats JSON:'), textarea, btn, output]))

### 📋 All Game Stats & Summary

In [ ]:
if spark.catalog.tableExists("game_stats"):
    # Check which columns exist in the table
    table_cols = set(c.name for c in spark.table("game_stats").schema.fields)
    has_new = "player_name" in table_cols

    if has_new:
        display(spark.sql("""
            SELECT player_name, timestamp, score, won, level_reached, duration_seconds,
                   deaths_total, deaths_lava, bosses_killed, coins_collected,
                   enemies_stomped, enemies_zapped, final_form
            FROM game_stats ORDER BY timestamp DESC
        """))

        print("\n📊 Summary by Player:")
        spark.sql("""
            SELECT
                player_name,
                COUNT(*) AS games_played,
                SUM(CASE WHEN won THEN 1 ELSE 0 END) AS wins,
                ROUND(SUM(CASE WHEN won THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS win_rate_pct,
                MAX(score) AS high_score,
                ROUND(AVG(score), 0) AS avg_score,
                MAX(level_reached) AS max_level,
                SUM(bosses_killed) AS total_bosses_killed,
                ROUND(SUM(duration_seconds), 1) AS total_playtime_sec
            FROM game_stats
            GROUP BY player_name
            ORDER BY high_score DESC
        """).show()
    else:
        display(spark.sql("""
            SELECT timestamp, score, won, duration_seconds,
                   deaths_total, coins_collected,
                   enemies_stomped, enemies_zapped, final_form
            FROM game_stats ORDER BY timestamp DESC
        """))

    print("\n📊 Overall Stats:")
    cols = [
        "COUNT(*) AS total_games",
        "MAX(score) AS all_time_high",
        "SUM(CASE WHEN won THEN 1 ELSE 0 END) AS total_wins",
        "SUM(deaths_total) AS total_deaths",
        "SUM(coins_collected) AS total_coins",
        "ROUND(SUM(duration_seconds) / 60, 1) AS total_playtime_min",
    ]
    if has_new:
        cols.insert(0, "COUNT(DISTINCT player_name) AS unique_players")
        cols.append("SUM(deaths_lava) AS total_lava_deaths")
        cols.append("SUM(bosses_killed) AS total_bosses_killed")
    spark.sql(f"SELECT {', '.join(cols)} FROM game_stats").show()
else:
    print("No game_stats table yet. Play a game and save the stats first!")